## Heaps / Priority Queue

A heap is a complete binary tree satisfying the heap property — every parent is smaller than its children (min-heap) or larger (max-heap). It's the most efficient way to repeatedly find and remove the minimum or maximum element.


Core Operations

heappush -> O(log n) -> Add element, bubble up

heappop -> O(log n) -> Remove min/max, bubble down

heapify -> O(n) -> Build heap from array

peek -> O(1) -> View min/max without removing

In [2]:
import heapq

heap = []
heapq.heappush(heap, 3)
heapq.heappush(heap, 1)
heapq.heappush(heap, 5)

print(heap[0])            # 1  → peek at minimum, O(1)
print(heapq.heappop(heap))  # 1  → remove minimum, O(log n)

# Build from existing list
nums = [3, 1, 4, 1, 5, 9, 2, 6]
heapq.heapify(nums)         # O(n) in-place

# Max-heap: negate values
max_heap = []
heapq.heappush(max_heap, -3)
heapq.heappush(max_heap, -1)
max_val = -heapq.heappop(max_heap)   # 3

1
1


##  Kth Largest / Smallest Element

Kth Largest — maintain a min-heap of size k. The top is always the kth largest.

In [ ]:
import heapq

def find_kth_largest(nums, k):
    heap = []
    for num in nums:
        heapq.heappush(heap, num)

        if len(heap)>k:
            heapq.heappop(heap)
    return heap[0]
    
print(find_kth_largest([3, 2, 1, 5, 6, 4], 3))


4


In [6]:
def find_kth_smallest(nums, k):
    max_heap = []
    for num in nums:
        heapq.heappush(max_heap, -num)
        if len(max_heap) > k:
            heapq.heappop(max_heap)
    return max_heap[0]
print(find_kth_smallest([3, 2, 1, 5, 6, 4], 3))

-3


In [9]:
def top_k_elements(nums, k):
    freq = {}
    for num in nums:
        freq[num] = freq.get(num, 0)+1
    min_heap = []
    for num, count in freq.items():
        heapq.heappush(min_heap, (count, num))
        if len(min_heap) > k:
            heapq.heappop(min_heap)
    return [num for count, num in min_heap]
print(top_k_elements([1, 1, 1, 2, 2, 3, 3], 2))

[3, 1]


### Merge K sorted Lists

Push the first element of each list into the heap with a pointer back to its list. Always pop the smallest and push the next element from the same list.

In [ ]:
def merge_k_sorted(lists: list[list[int]]) -> list[int]:
    min_heap = []
    result = []

    # Seed heap with first element of each list
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(min_heap, (lst[0], i, 0))
                                    # (value, list_idx, element_idx)
    print(min_heap)
    while min_heap:
        val, list_idx, elem_idx = heapq.heappop(min_heap)
        result.append(val)

        # Push next element from the same list
        next_idx = elem_idx + 1
        if next_idx < len(lists[list_idx]):
            next_val = lists[list_idx][next_idx]
            heapq.heappush(min_heap, (next_val, list_idx, next_idx))
        
    return result

merge_k_sorted([[1,4,7],[2,5,8],[3,6,9]])
# → [1,2,3,4,5,6,7,8,9]

[(1, 0, 0), (2, 1, 0), (3, 2, 0)]
[(2, 1, 0), (3, 2, 0), (4, 0, 1)]
[(3, 2, 0), (4, 0, 1), (5, 1, 1)]
[(4, 0, 1), (5, 1, 1), (6, 2, 1)]
[(5, 1, 1), (6, 2, 1), (7, 0, 2)]
[(6, 2, 1), (7, 0, 2), (8, 1, 2)]
[(7, 0, 2), (8, 1, 2), (9, 2, 2)]
[(8, 1, 2), (9, 2, 2)]
[(9, 2, 2)]
[]


[1, 2, 3, 4, 5, 6, 7, 8, 9]

## Task Scheduling / Meeting Rooms

Meeting Rooms II — minimum number of rooms needed.

In [81]:
intervals = [[0, 30], [15, 20], [5, 10]]
def min_meeting_rooms(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda k: k[0])
    print(intervals)
    min_heap = []
    for start, end in intervals:
        if min_heap and min_heap[0] <= start:
            heapq.heapreplace(min_heap, end)
        else:
             heapq.heappush(min_heap, end)
    print(min_heap)
    return len(min_heap)
print(min_meeting_rooms(intervals))

[[0, 30], [5, 10], [15, 20]]
[20, 30]
2


### K Closest Points to Origin

In [ ]:
def k_closest(points, k):
    max_heap = []
    for x, y in points:
        dist = -((x * x) + (y * y))
        heapq.heappush(max_heap, (dist, x, y))
        if len(max_heap) > k:
            print(max_heap[0])
            heapq.heappop(max_heap)
    print(max_heap)

k_closest([[1, 3], [-2, 2], [5, 8], [0, 1]], 2)

(-89, 5, 8)
(-10, 1, 3)
[(-8, -2, 2), (-1, 0, 1)]


### Median From Data Stream

Maintain two heaps — a max-heap for the lower half and a min-heap for the upper half. Keep them balanced.

In [ ]:
class MedianFinder:
    def __init__(self):
        self.lower = []    # max-heap (negate values) — smaller half
        self.upper = []    # min-heap — larger half

    def add_num(self, num: int) -> None:
        # Always push to lower first
        heapq.heappush(self.lower, -num)

        # Balance: lower's max must be ≤ upper's min
        if self.lower and self.upper and (-self.lower[0] > self.upper[0]):
            heapq.heappush(self.upper, -heapq.heappop(self.lower))

        # Balance sizes: lower can have at most 1 extra
        if len(self.lower) > len(self.upper) + 1:
            heapq.heappush(self.upper, -heapq.heappop(self.lower))
        elif len(self.upper) > len(self.lower):
            heapq.heappush(self.lower, -heapq.heappop(self.upper))

    def find_median(self) -> float:
        if len(self.lower) > len(self.upper):
            return float(-self.lower[0])
        return (-self.lower[0] + self.upper[0]) / 2.0